# Phase 3 — Models

Builds two architectures, per [DECISIONS.md](../DECISIONS.md) #5 and #6:
1. **CSRNet**: VGG-16/ImageNet front end (frozen pretrained features) + dilated-convolution back end, fine-tuned on ShanghaiTech.
2. **A small CNN trained from scratch** — the Decision #5 comparison point, same input/output shapes so it's a fair comparison.

This notebook only builds and shape-checks both models with a forward pass on one real cached sample from Phase 1. Training happens in Phase 4.

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from pathlib import Path

print("TensorFlow", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))

PROCESSED_DIR = Path("../data/processed/part_A")
train_df = pd.read_csv(PROCESSED_DIR / "manifest_train.csv")
test_df = pd.read_csv(PROCESSED_DIR / "manifest_test.csv")
print(f"train: {len(train_df)} images, test: {len(test_df)} images")


## 1. Load one known sample

Using `IMG_1.jpg` again (704×1024, both divisible by 8) — same image Lessons 0.2–0.3 and Phase 1 already verified, so any shape mismatch here is the model's fault, not the data's.

In [ ]:
from PIL import Image

sample_row = test_df[test_df["image_path"].str.contains("IMG_1.jpg", regex=False)].iloc[0]
sample_img = np.array(Image.open(sample_row["image_path"])).astype(np.float32)
sample_density = np.load(sample_row["density_map_path"])

print("image shape:", sample_img.shape)
print("density map shape:", sample_density.shape)
print("stated count:", sample_row["stated_count"])
print("density sum:", round(sample_density.sum(), 2))

# VGG16 preprocessing expects a batch dimension and ImageNet channel preprocessing
sample_batch = np.expand_dims(sample_img, axis=0)
sample_batch = keras.applications.vgg16.preprocess_input(sample_batch)
print("model input batch shape:", sample_batch.shape)


## 2. CSRNet front end: VGG-16 truncated at `block4_conv3`

Per the CSRNet paper: keep VGG-16's first 10 conv layers (through `block4_conv3`) and 3 of its 4 max-pools, discard the 4th pool, the entire `block5` stage, and the fully-connected classifier head. That leaves 3 poolings — an 8× spatial downsample — with 512 output channels, still carrying ImageNet-pretrained weights.

In [ ]:
def build_vgg16_frontend(input_shape=(None, None, 3)):
    vgg = keras.applications.VGG16(include_top=False, weights="imagenet", input_shape=input_shape)
    frontend = keras.Model(
        inputs=vgg.input,
        outputs=vgg.get_layer("block4_conv3").output,
        name="csrnet_frontend_vgg16",
    )
    return frontend


frontend = build_vgg16_frontend()
frontend.summary()


## 3. CSRNet back end: dilated convolutions

6 dilated 3×3 convolutions (dilation rate 2, configuration B from the paper — the one used for ShanghaiTech Part A), channels 512→512→512→256→128→64, no further pooling, then a 1×1 conv down to a single-channel density map. Dilation grows the receptive field without shrinking resolution further — output stays at the front end's 8×-downsampled size.

In [ ]:
BACKEND_CHANNELS = [512, 512, 512, 256, 128, 64]


def build_csrnet(input_shape=(None, None, 3)):
    frontend = build_vgg16_frontend(input_shape)
    x = frontend.output
    for i, channels in enumerate(BACKEND_CHANNELS):
        x = layers.Conv2D(
            channels, kernel_size=3, dilation_rate=2, padding="same",
            activation="relu", name=f"backend_conv{i+1}",
        )(x)
    density_map = layers.Conv2D(1, kernel_size=1, padding="same", name="density_output")(x)

    model = keras.Model(inputs=frontend.input, outputs=density_map, name="CSRNet")
    return model


csrnet = build_csrnet()
csrnet.summary()


## 4. Forward pass on the real sample

This is still an **untrained** model (random back end + pretrained-but-not-fine-tuned front end), so the predicted density values are meaningless. This cell only checks that the architecture runs end-to-end and the output shape is what the 8× downsample predicts.

In [ ]:
pred = csrnet(sample_batch, training=False)
expected_h, expected_w = sample_img.shape[0] // 8, sample_img.shape[1] // 8

print("input shape:", sample_batch.shape)
print("predicted density map shape:", pred.shape)
print("expected (input / 8):", (1, expected_h, expected_w, 1))
assert pred.shape[1] == expected_h and pred.shape[2] == expected_w
print("Output shape matches the 8x downsample, as expected.")
print("predicted sum (meaningless, untrained):", float(tf.reduce_sum(pred)))


## 5. Downsample the ground-truth density map to match

The cached density maps (Phase 1) are full input resolution; the model outputs 1/8 resolution. Training needs the target at the same resolution as the prediction. Pool each 8×8 block by **sum**, not average — average would shrink the total by 64×, sum keeps the same sum-preservation invariant verified back in Lesson 0.3.

In [ ]:
def downsample_density_map(density_map, factor=8):
    """Sum-pool a density map by `factor`. Requires both dims divisible by factor
    (Phase 4's patch-based training will guarantee this by construction)."""
    h, w = density_map.shape
    assert h % factor == 0 and w % factor == 0, "dims must be divisible by factor"
    x = tf.constant(density_map.reshape(1, h, w, 1), dtype=tf.float32)
    pooled = tf.nn.avg_pool2d(x, ksize=factor, strides=factor, padding="VALID")
    pooled = pooled * (factor * factor)  # undo the averaging to recover block sums
    return pooled[0, :, :, 0].numpy()


downsampled_gt = downsample_density_map(sample_density)
print("original density map shape:", sample_density.shape, "sum:", round(sample_density.sum(), 4))
print("downsampled shape:", downsampled_gt.shape, "sum:", round(downsampled_gt.sum(), 4))
assert downsampled_gt.shape == (expected_h, expected_w)
assert abs(downsampled_gt.sum() - sample_density.sum()) < 0.01
print("Downsampling preserves the sum, and matches the model's output shape.")


## 6. From-scratch CNN (Decision #5's comparison point)

No pretrained weights, no dilation — just 3 conv+pool blocks (same 8× downsample as CSRNet's front end, for a fair comparison) followed by a couple of refining convs and the same 1×1 output head. Much smaller capacity, trained from random initialization.

In [ ]:
def build_scratch_cnn(input_shape=(None, None, 3)):
    inputs = keras.Input(shape=input_shape)
    x = inputs
    for channels in [24, 48, 96]:  # 3 blocks -> same 8x downsample as the VGG16 front end
        x = layers.Conv2D(channels, 3, padding="same", activation="relu")(x)
        x = layers.Conv2D(channels, 3, padding="same", activation="relu")(x)
        x = layers.MaxPooling2D(2)(x)
    x = layers.Conv2D(64, 3, padding="same", activation="relu")(x)
    x = layers.Conv2D(32, 3, padding="same", activation="relu")(x)
    density_map = layers.Conv2D(1, 1, padding="same", name="density_output")(x)
    return keras.Model(inputs=inputs, outputs=density_map, name="ScratchCNN")


scratch_cnn = build_scratch_cnn()
scratch_cnn.summary()


In [ ]:
scratch_pred = scratch_cnn(sample_batch, training=False)
print("ScratchCNN output shape:", scratch_pred.shape)
assert scratch_pred.shape[1] == expected_h and scratch_pred.shape[2] == expected_w
print("Matches CSRNet's output shape, as expected.")

print()
csrnet_trainable = sum(np.prod(v.shape) for v in csrnet.trainable_variables)
csrnet_total = csrnet.count_params()
scratch_total = scratch_cnn.count_params()

print(f"CSRNet: {csrnet_total:,} total params ({csrnet_trainable:,} trainable "
      f"as built, before any freezing decision)")
print(f"ScratchCNN: {scratch_total:,} total params, all trainable, all random init")
print(f"CSRNet has {csrnet_total / scratch_total:.1f}x more parameters")


## Summary

Both models build, run a forward pass on a real cached sample, and produce the correctly-shaped (1/8-resolution, single-channel) density map output. The ground-truth downsampling function matches that resolution while preserving the count-sum invariant from Lesson 0.3.

Neither model is trained yet — both are producing meaningless output right now (random back end for CSRNet, fully random init for ScratchCNN). Phase 4 is training both on the full cached dataset from Phase 1, then evaluating MAE on the test set.

**Open question for Phase 4:** whether to freeze the VGG-16 front end entirely or fine-tune it at a lower learning rate — the CSRNet paper fine-tunes it; given CPU-only training here, freezing it first (cheaper, front end only needs one forward pass, no backprop through it) is worth trying before committing to full fine-tuning.